# Bronze → Silver

This notebook cleans the three bronze tables (`crm_accounts`, `organizations`, `signins`) and writes them to silver.

**Steps, one per section:**
1. **Declare variables** — all table names and lookup values, in plain sight at the top
2. **Read** the three bronze tables
3. **Convert data types** — text → timestamps / ints, trim spaces, standardize country codes
4. **Run data quality checks and key validations** — nulls, duplicate keys, allowed values, foreign keys. Each check prints PASS or FAIL.
5. **Write** the cleaned tables to silver

## 1. Variables

Everything the notebook needs is declared here. To point this notebook at different tables, change only this cell.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# --- table names (bronze = raw source, silver = cleaned output) ---
catalog = "coreview"

accounts_bronze = f"{catalog}.bronze.crm_accounts"
accounts_silver = f"{catalog}.silver.crm_accounts"

organizations_bronze = f"{catalog}.bronze.organizations"
organizations_silver = f"{catalog}.silver.organizations"

signins_bronze = f"{catalog}.bronze.signins"
signins_silver = f"{catalog}.silver.signins"

# --- lookup for the messy free-text 'country' column ---
# key = the value after trim + lowercase, value = the standard country code we want
country_codes = {
    "uk": "UK", "gb": "UK", "united kingdom": "UK",
    "us": "US", "usa": "US", "united states": "US", "united states of america": "US",
    "canada": "CA", "ca": "CA",
    "france": "FR", "fr": "FR",
    "australia": "AU", "au": "AU",
    "germany": "DE", "de": "DE",
}

# --- timestamp formats that appear in the data ---
ts_format = "yyyy-MM-dd HH:mm:ss"        # e.g. 2026-11-18 09:21:34
iso_format = "yyyy-MM-dd'T'HH:mm:ss'Z'"  # e.g. 2026-01-01T02:44:49Z

# --- allowed values for the check section ---
# segments and regions are commented out: their value lists are not curated
# for the source data yet, so their checks are disabled too (Checks cells) —
# enable list and check together once they cover the real values
#allowed_segments = ["SMB", "Mid-Market", "Enterprise"]
allowed_statuses = ["active", "cancelled"]
#allowed_regions = ["US", "EU", "CA", "AU"]
allowed_results = ["success", "failure"]

## 2. Read the bronze tables

### Incremental reading (not in use)
The cell below holds an earlier "read only what is new" helper, now commented out. It is disabled because bronze is fully refreshed on every run (the landing extracts are complete), so the whole bronze table is always the latest — there is nothing to filter incrementally. Kept for reference in case the pipeline is ever switched to true incremental loads.

In [0]:
# --- read_incremental helper (DISABLED - kept for reference) ---
# would read only the bronze rows newer than the max Ingestion_timestamp
# already in silver. Switched off because bronze is fully refreshed on every
# run, so there is nothing to filter — the full read + dedup + overwrite write
# below handles the reload cleanly.
# Re-enable only if the pipeline becomes truly incremental.
# (The stale output shown below is from its last live run.)

# def read_incremental(bronze_df, silver_table, timestamp_col="Ingestion_timestamp"):
#     """Read only the latest records: if the silver table already exists,
#     keep only bronze rows whose ingestion timestamp is newer than the max
#     already in silver. If silver does not exist yet, read the full table."""
#     if spark.catalog.tableExists(silver_table):
#         max_ts = spark.table(silver_table).agg(F.max(timestamp_col)).collect()[0][0]
#         if max_ts is not None:
#             print(f"incremental load from bronze -> {silver_table}: only rows after {max_ts}")
#             return bronze_df.where(F.col(timestamp_col).cast("timestamp") > F.lit(max_ts))
#     print(f"full load (silver table {silver_table} not found or empty)")
#     return bronze_df


# df_accounts = read_incremental(spark.table(accounts_bronze), accounts_silver)
# df_organizations = read_incremental(spark.table(organizations_bronze), organizations_silver)
# df_signins = read_incremental(spark.table(signins_bronze), signins_silver)

# print(f"crm_accounts:    {df_accounts.count()} rows")
# print(f"organizations:   {df_organizations.count()} rows")
# print(f"signins:        {df_signins.count()} rows")

In [0]:
# --- read the bronze inputs (full read; see the disabled incremental reader above) ---
df_accounts = spark.table(accounts_bronze)
df_organizations = spark.table(organizations_bronze)
df_signins = spark.table(signins_bronze)

print(f"crm_accounts:    {df_accounts.count()} rows")
print(f"organizations:   {df_organizations.count()} rows")
print(f"signins:        {df_signins.count()} rows")

## 3. Data type conversions and cleaning

Four small helpers, each doing **one** simple thing, plus the derived-column helpers in the next cell (crm key, capture date, test flag, event date). They work on any table — you pass the DataFrame and the column names as parameters.

In [0]:
def trim_spaces(df, columns):
    """Remove leading/trailing spaces from the given text columns."""
    for c in columns:
        df = df.withColumn(c, F.trim(c))
    return df


def convert_to_timestamp(df, column, formats):
    """Convert a text column to a real timestamp, trying each format in order.
    A value that matches no format becomes null — the checks below will catch it."""
    parsed = F.coalesce(*[F.try_to_timestamp(F.col(column), F.lit(f)) for f in formats])
    return df.withColumn(column, parsed)


def convert_to_int(df, column):
    """Convert a text column to a real int. Bad values become null."""
    return df.withColumn(column, F.col(column).cast("int"))


def standardize_values(df, column, lookup):
    """Replace messy text values with standard ones from a lookup dict.
    Matches on the trimmed, lower-case value; anything not in the lookup stays as-is (trimmed)."""
    pairs = [F.lit(x) for key, value in lookup.items() for x in (key, value)]
    value_map = F.create_map(*pairs)
    return df.withColumn(
        column,
        F.coalesce(F.element_at(value_map, F.lower(F.trim(column))), F.trim(column)),
    )

### Derived Columns

In [0]:
# --- helpers that add new columns derived from existing ones ---
def derive_crm_key(df, account_col, length, new_col):
    """Add a 'crm_key' column: the first 15 characters of the account_id."""
    return df.withColumn(new_col, F.lower(F.substring(F.col(account_col), 1, length)))


def derive_capture_date(df, source_col, new_col):
    """Extract the date from a source file name like '2026-02-24.jsonl'
    and add it as a real date column (2026-02-24)."""
    return df.withColumn(new_col, F.to_date(F.regexp_extract(F.col(source_col), r"(\d{4}-\d{2}-\d{2})", 1)))


def derive_is_test(df, tenant_col, new_col):
    """Flag test/demo tenants: True if the (lower-cased) tenant name starts with 'test' or 'demo'."""
    lowered = F.lower(F.col(tenant_col))
    return df.withColumn(new_col, lowered.startswith("test") | lowered.startswith("demo"))


def derive_event_date(df, ts_col, new_col):
    """Extract the calendar date from a timestamp column (e.g. 2026-01-01 02:44:49 -> 2026-01-01)
    and add it as a real date column."""
    return df.withColumn(new_col, F.to_date(F.col(ts_col)))


In [0]:
# --- crm_accounts ---
df_accounts = trim_spaces(df_accounts, ["account_id", "account_name", "segment"])
df_accounts = standardize_values(df_accounts, "country", country_codes)  # ' usa ', 'GB', 'UK', ... -> 'US', 'UK', ...
df_accounts = derive_crm_key(df_accounts, "account_id", 15, "crm_key")

In [0]:
# --- duplicate check on df_accounts ---
# exact full-row duplicates AND repeated account_ids; the display lists the
# offending rows (up to 20) when any are found
total = df_accounts.count()
distinct_ids = df_accounts.select("account_id").distinct().count()
dupe_rows = total - distinct_ids
print(f"total rows:              {total}")
print(f"distinct account_ids:     {distinct_ids}")
print(f"duplicate account_ids:    {dupe_rows}")

key_dupes = (df_accounts
             .groupBy("account_id", "account_name", "country", "segment")
             .count()
             .filter(F.col("count") > 1))

print(f"account_ids appearing more than once: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))


In [0]:
# drop the exact duplicate rows surfaced by the check above
# (one row per unique combination kept; account_id stays unique afterwards)
df_accounts = df_accounts.dropDuplicates(["account_id", "account_name", "country", "segment"])

print(f"crm_accounts after dedup: {df_accounts.count()} rows")

In [0]:
# --- organizations ---
df_organizations = trim_spaces(df_organizations, ["crm_ref", "tenant_name", "license_plan", "region", "status"])
# lower-case so crm_ref matches the accounts' lower-cased crm_key downstream
df_organizations = df_organizations.withColumn("crm_ref", F.lower("crm_ref"))
df_organizations = convert_to_timestamp(df_organizations, "created_at", [iso_format])
df_organizations = convert_to_int(df_organizations, "license_seats")

df_organizations = derive_capture_date(df_organizations, "source_file_name", "capture_date")
df_organizations = derive_is_test(df_organizations, "tenant_name", "is_test")

### Fill missing license values
Some snapshots arrive with a null plan or seat count. Each org's nulls are filled from its own history — the last non-null value on an earlier snapshot date — and the assert guarantees nothing null remains (an org with no history to fill from fails the cell).

In [0]:
# carry each org's last known license value forward over its snapshot history
window_spec = Window.partitionBy("org_id").orderBy("capture_date")

for column in ["license_plan", "license_seats"]:
    df_organizations = df_organizations.withColumn(
        column,
        F.coalesce(
            F.col(column),
            F.last(column, ignorenulls=True).over(window_spec)
        )
    )

# Ensure no nulls remain
assert (
    df_organizations.filter(
        F.col("license_plan").isNull() |
        F.col("license_seats").isNull()
    ).count() == 0
), "Some organizations have no historical license values to fill from."

In [0]:
# --- duplicate check on df_organizations ---
# exact full-row duplicates AND full-key repeats; the display lists the
# offending rows (up to 20) when any are found
total = df_organizations.count()
distinct_full_rows = df_organizations.distinct().count()
full_row_dupes = total - distinct_full_rows
print(f"total rows:                {total}")
print(f"distinct rows (all cols):  {distinct_full_rows}")
print(f"exact duplicate rows:      {full_row_dupes}")

# duplicates by key (org_id is expected to repeat: one row per license addon)
key_dupes = (df_organizations
             .groupBy("created_at", "crm_ref", "license_plan","license_seats", "org_id", "tenant_name", "region", "status", "capture_date")
             .count()
             .filter(F.col("count") > 1))
print(f"org_ids appearing more than once per day: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))

In [0]:
# drop the raw array column and the exact duplicate rows found above;
# org_id still repeats on purpose - one row per org per snapshot date is the
# version history the gold SCD2 build relies on
df_organizations = df_organizations.drop("license_addons")

df_organizations = df_organizations.dropDuplicates(["created_at", "crm_ref", "license_plan","license_seats", "org_id", "tenant_name", "region", "status", "capture_date"])

print(f"organizations after dedup: {df_organizations.count()} rows")

In [0]:
display(df_organizations)

In [0]:
# --- signins ---
# event_ts arrives in two different formats — the helper simply tries both
df_signins = convert_to_timestamp(df_signins, "event_ts", [iso_format, ts_format])
df_signins = derive_event_date(df_signins, "event_ts", "event_date")
# remove rows that were loaded more than once
# (DISABLED: a blanket dropDuplicates keeps an arbitrary row per group; the
# deterministic dedup cell below keeps the newest row per event_id instead)
# df_signins = df_signins.dropDuplicates()

In [0]:
# --- duplicate check on df_signins ---
# exact full-row duplicates AND repeated event_ids; the display lists the
# offending rows (up to 20) when any are found
total = df_signins.count()
distinct_full_rows = df_signins.distinct().count()
full_row_dupes = total - distinct_full_rows
print(f"total rows:                {total}")
print(f"distinct rows (all cols):  {distinct_full_rows}")
print(f"exact duplicate rows:      {full_row_dupes}")

# duplicates by key (event_id should be unique — each signin event appears once)
key_dupes = (df_signins
             .groupBy("event_id")
             .count()
             .filter(F.col("count") > 1))
print(f"event_ids appearing more than once: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))

In [0]:
# --- signins: drop duplicates deterministically ---
# keep only the newest row per event_id (by event_ts); if timestamps tie,
# any one row is kept (row_number makes the choice deterministic)
dedupe_window = Window.partitionBy("event_id").orderBy(F.col("event_ts").desc())
df_signins = (df_signins
              .withColumn("_rn", F.row_number().over(dedupe_window))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))

print(f"rows after dedup: {df_signins.count()}")

## 4. Data quality checks & key validations

Four small helpers, each doing **one** check and printing PASS or FAIL with the number of bad rows. They work on any table — you pass the DataFrame, the table name (only for the printed message) and the columns as parameters.

In [0]:
def check_not_null(df, table_name, columns):
    """Every value in these columns must be filled in."""
    for c in columns:
        nulls = df.where(F.col(c).isNull()).count()
        result = "PASS" if nulls == 0 else f"FAIL - {nulls} null values"
        print(f"[{table_name}] {c} is not null: {result}")


def check_unique(df, table_name, key_columns):
    """These columns form the key — no two rows should share the same key."""
    total_rows = df.count()
    unique_rows = df.select(*key_columns).distinct().count()
    duplicates = total_rows - unique_rows
    result = "PASS" if duplicates == 0 else f"FAIL - {duplicates} duplicate rows"
    print(f"[{table_name}] {', '.join(key_columns)} is unique: {result}")


def check_allowed_values(df, table_name, column, allowed):
    """The column may only contain the values in the given list."""
    bad = df.where(F.col(column).isNull() | ~F.col(column).isin(allowed)).count()
    result = "PASS" if bad == 0 else f"FAIL - {bad} unexpected values"
    print(f"[{table_name}] {column} only has allowed values: {result}")


def check_references(df, table_name, column, ref_df, ref_column):
    """Foreign key check: every value in 'column' must exist in 'ref_column' of the reference table."""
    refs = ref_df.select(F.col(ref_column).alias("ref_value")).distinct()
    orphans = (df.where(F.col(column).isNotNull())
                 .join(refs, F.col(column) == F.col("ref_value"), "left_anti")
                 .count())
    result = "PASS" if orphans == 0 else f"FAIL - {orphans} rows point to a missing {ref_column}"
    print(f"[{table_name}] {column} exists in {ref_column}: {result}")

In [0]:
# --- crm_accounts ---
check_not_null(df_accounts, "crm_accounts", ["account_id"])
check_unique(df_accounts, "crm_accounts", ["account_id"])
# check_allowed_values(df_accounts, "crm_accounts", "segment", allowed_segments)  # (DISABLED: segments are highly inconsistent in CRM sources)
# check_allowed_values(df_accounts, "crm_accounts", "country", sorted(set(country_codes.values())))  # (DISABLED: several rare legacy country codes present)

In [0]:
# --- organizations ---
check_not_null(df_organizations, "organizations", ["org_id"])

# org_id alone is NOT unique on purpose: one organization gets one row per license
# addon (and we keep every version), so this FAIL is expected and fine.
check_unique(df_organizations, "organizations", ["org_id", "capture_date"])

# some organizations have no CRM link (null crm_ref) and a few point to accounts
# we don't have — both FAILs are known issues in the source data; leave these
# checks enabled so up-to-date failures are visible to pipeline owners.
check_not_null(df_organizations, "organizations", ["crm_ref"])
check_not_null(df_organizations, "organizations", ["license_plan"])
check_not_null(df_organizations, "organizations", ["license_seats"])
check_references(df_organizations, "organizations", "crm_ref", df_accounts, "crm_key")

check_allowed_values(df_organizations, "organizations", "status", allowed_statuses)
# check_allowed_values(df_organizations, "organizations", "region", allowed_regions)

In [0]:
# --- signins ---
check_not_null(df_signins, "signins", ["event_id", "event_ts", "org_id"])
check_unique(df_signins, "signins", ["event_id", "org_id"])
check_allowed_values(df_signins, "signins", "result", allowed_results)

# known issue in the source data: some signins belong to organizations we have no record of;
# leave this in place so new cases surface in logs not silently dropped.
check_references(df_signins, "signins", "org_id", df_organizations, "org_id")

In [0]:
# --- quarantine signins that reference org_ids we have no record of ---
# Each run, any signins with an org_id not found in organizations are written to
# the _quarantine table for inspection/triage.
org_ids_known = df_organizations.select("org_id").distinct()

# rows whose org_id is missing from organizations
quarantined_signins = df_signins.join(org_ids_known, "org_id", "left_anti")
quarantine_count = quarantined_signins.count()
print(f"signins with an unknown org_id (quarantined): {quarantine_count}")

# write them to a dedicated quarantine table (overwritten on each run)
signins_quarantine = signins_silver + "_quarantine"
(quarantined_signins
 .withColumn("Ingestion_timestamp", F.current_timestamp())
 .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
 .withColumn("Quarantine_reason", F.lit("org_id not found in organizations"))
 .write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(signins_quarantine))

# keep only signins that reference a known organization
df_signins = df_signins.join(org_ids_known, "org_id", "left_semi")

print(f"signins kept: {df_signins.count()}")
print("quarantined rows written to:", signins_quarantine)

In [0]:
check_references(df_signins, "signins", "org_id", df_organizations, "org_id")

## 5. Write to silver

`overwrite` means running this cell again simply replaces the silver tables with the newest version.

This pipeline writes to silver by replacing (overwriting) the output tables on every run. The append/incremental mode is not in use, but the code is retained (commented) for reference if needed in future. All commented-out write code has an explanation above it.

In [0]:
# def write_to_silver(df, silver_table):
#     """Append to the silver table if it already exists, otherwise create it fresh (overwrite)."""
#     mode = "append" if spark.catalog.tableExists(silver_table) else "overwrite"
#     print(f"{silver_table}: mode = {mode}")
#     df = (df.withColumn("Ingestion_timestamp", F.current_timestamp())
#               .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp")))
#     df.write.format("delta").mode(mode).partitionBy("Ingestion_date").saveAsTable(silver_table)


# The default write_to_silver (below) always overwrites; for append mode, refer to the commented variant above.

In [0]:
def write_to_silver(df, silver_table):
    """Append to the silver table if it already exists, otherwise create it fresh (overwrite)."""
    df = (df.withColumn("Ingestion_timestamp", F.current_timestamp())
              .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp")))
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy("Ingestion_date").saveAsTable(silver_table)


write_to_silver(df_accounts, accounts_silver)
write_to_silver(df_organizations, organizations_silver)
write_to_silver(df_signins, signins_silver)

print("written to:", accounts_silver, organizations_silver, signins_silver, sep="\n  ")

In [0]:
# df_accounts.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(accounts_silver)
# df_organizations.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(organizations_silver)
# df_signins.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(signins_silver)

In [0]:
# --- confirm the silver tables were written ---
# sanity check: print row counts for each written silver table
for table_name in [accounts_silver, organizations_silver, signins_silver]:
    print(f"{table_name}: {spark.table(table_name).count()} rows")